# XTTS-v2 smoke test (Colab)
Usa exclusivamente tres muestras del split `dev` reproducible de FLEURS. No emite juicios subjetivos.

In [ ]:
import platform, subprocess, sys, importlib.metadata as md
print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=False)
for name in ('torch','torchaudio','transformers','numpy','coqui-tts','speechbrain'):
    try: print(f'{name}={md.version(name)}')
    except md.PackageNotFoundError: print(f'{name}=NOT INSTALLED')

In [ ]:
# Entorno XTTS separado/reproducible. Ejecutar antes de importar torch.
%pip install -q 'torch==2.8.0' 'torchaudio==2.8.0' 'coqui-tts==0.27.5' 'transformers==4.57.3' 'numpy==2.4.6' 'speechbrain==1.1.1' 'soundfile==0.14.0' 'scipy==1.17.1' 'datasets==3.6.0' 'pyarrow==18.1.0' 'pandas==2.3.3' 'pyyaml==6.0.3'
print('Instalación terminada. Si Colab solicita reiniciar, reinicie y ejecute Run all de nuevo.')

In [ ]:
import os, sys, subprocess, time, json
from pathlib import Path
REPO_URL='https://github.com/Setrox1250/Proyecto-IA-Traductor.git'; ROOT=Path('/content/Proyecto-IA-Traductor')
if not ROOT.exists(): subprocess.run(['git','clone',REPO_URL,str(ROOT)],check=True)
os.chdir(ROOT); sys.path.insert(0,str(ROOT))
import torch, soundfile as sf
from src.dataset.fleurs import stream_fleurs, take_samples, build_fleurs_manifest
from src.dataset.manifest import load_fleurs_english_references, apply_fleurs_references, assign_splits
from src.voice_cloning.audio import prepare_reference_audio
from src.voice_cloning.synthesize import synthesize
# Reconstruye Dataset B si no se montó previamente; split con seed=42: 14 dev / 6 test.
manifest_path=ROOT/'data/processed/manifests/evaluation_v1.csv'
if manifest_path.exists():
    import pandas as pd; manifest=pd.read_csv(manifest_path)
else:
    manifest=build_fleurs_manifest(take_samples(stream_fleurs(),n=20,seed=42),audio_dir=ROOT/'data/raw/audio')
    manifest=apply_fleurs_references(manifest,load_fleurs_english_references())
    manifest=assign_splits(manifest,seed=42); manifest_path.parent.mkdir(parents=True,exist_ok=True); manifest.to_csv(manifest_path,index=False)
samples=manifest.loc[manifest['split']=='dev'].head(3).to_dict('records')
assert len(samples)==3, 'Se requieren tres muestras DEV.'
out=ROOT/'outputs/smoke_tests/xtts'; out.mkdir(parents=True,exist_ok=True); results=[]
for i,row in enumerate(samples):
    source=ROOT/row['audio_path']; reference=out/f'reference_{i:02d}.wav'; output=out/f'sample_{i:02d}.wav'
    prepare_reference_audio(source,reference); torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    t0=time.perf_counter(); synthesize(str(row['translation_en']),str(reference),str(output),language='en'); elapsed=time.perf_counter()-t0
    info=sf.info(output); results.append({'sample_id':row['sample_id'],'source_audio':str(source),'reference_sample_rate':sf.info(reference).samplerate,'text_en':row['translation_en'],'text_source':'FLEURS official English reference','output_audio':str(output),'output_sample_rate':info.samplerate,'output_duration':len(sf.read(output)[0])/info.samplerate,'inference_time_sec':elapsed,'GPU':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU','peak_vram_mb':torch.cuda.max_memory_allocated()/2**20 if torch.cuda.is_available() else None})
(out/'results.json').write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding='utf-8'); results

In [ ]:
from IPython.display import Audio, display
for row in results:
    print(row['sample_id'], row['output_sample_rate'], 'Hz', f"{row['inference_time_sec']:.2f}s")
    display(Audio(row['source_audio'])); display(Audio(row['output_audio']))